In [1]:
#importing dataset

import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\Preetham.Reddy\Retail-Sales-Data\data\raw\orders.csv')
df

,order_id,order_date,brand,model_name,category,gender,size,color,base_price_usd,discount_percent,final_price_usd,units_sold,revenue_usd,payment_method,sales_channel,country,customer_income_level,customer_rating
0,ORD100000,2021-01-30,ASICS,Model-370,Running,Unisex,8,Black,162,15,137.70,1,137.70,Card,Retail Store,Germany,Low,4.6
1,ORD100001,2026-10-05,Reebok,Model-314,Lifestyle,Men,8,Grey,80,5,76.00,3,228.00,Card,Online,USA,Low,3.9
2,ORD100002,2023-11-12,ASICS,Model-763,Lifestyle,Men,8,Black,176,15,149.60,4,598.40,Cash,Retail Store,India,Medium,3.0
3,ORD100003,2026-08-29,Reebok,Model-905,Basketball,Women,7,White,61,15,51.85,2,103.70,Card,Retail Store,India,High,3.4
4,ORD100004,2019-11-09,Nike,Model-413,Training,Men,11,Black,80,0,80.00,4,320.00,Cash,Online,USA,Medium,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,ORD129995,2018-05-28,Puma,Model-892,Gym,Women,6,Blue,140,10,126.00,1,126.00,Card,Online,UAE,High,4.3
29996,ORD129996,2018-04-28,Adidas,Model-982,Running,Women,9,Grey,113,15,96.05,1,96.05,Card,Online,UK,High,3.5
29997,ORD129997,2026-02-17,ASICS,Model-463,Basketball,Men,11,Blue,130,30,91.00,1,91.00,Wallet,Online,India,Medium,4.4
29998,ORD129998,2024-02-21,New Balance,Model-984,Running,Unisex,11,White,186,10,167.40,2,334.80,Wallet,Retail Store,USA,High,3.9


In [2]:
#section1 : data quality and validation

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_id               30000 non-null  object 
 1   order_date             30000 non-null  object 
 2   brand                  30000 non-null  object 
 3   model_name             30000 non-null  object 
 4   category               30000 non-null  object 
 5   gender                 30000 non-null  object 
 6   size                   30000 non-null  int64  
 7   color                  30000 non-null  object 
 8   base_price_usd         30000 non-null  int64  
 9   discount_percent       30000 non-null  int64  
 10  final_price_usd        30000 non-null  float64
 11  units_sold             30000 non-null  int64  
 12  revenue_usd            30000 non-null  float64
 13  payment_method         30000 non-null  object 
 14  sales_channel          30000 non-null  object 
 15  co

In [4]:
#converting order_date column  to date format

df['order_date_parsed'] = pd.to_datetime(df['order_date'])

In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   order_id               30000 non-null  object        
 1   order_date             30000 non-null  object        
 2   brand                  30000 non-null  object        
 3   model_name             30000 non-null  object        
 4   category               30000 non-null  object        
 5   gender                 30000 non-null  object        
 6   size                   30000 non-null  int64         
 7   color                  30000 non-null  object        
 8   base_price_usd         30000 non-null  int64         
 9   discount_percent       30000 non-null  int64         
 10  final_price_usd        30000 non-null  float64       
 11  units_sold             30000 non-null  int64         
 12  revenue_usd            30000 non-null  float64       
 13  p

In [6]:
df[["order_date","order_date_parsed"]]

,order_date,order_date_parsed
0,2021-01-30,2021-01-30
1,2026-10-05,2026-10-05
2,2023-11-12,2023-11-12
3,2026-08-29,2026-08-29
4,2019-11-09,2019-11-09
...,...,...
29995,2018-05-28,2018-05-28
29996,2018-04-28,2018-04-28
29997,2026-02-17,2026-02-17
29998,2024-02-21,2024-02-21


In [7]:
invalid_date_count = df["order_date_parsed"].isna().sum()
invalid_date_count

0

In [8]:
today = pd.Timestamp.today().normalize()
#today
future_date_count = (df["order_date_parsed"]>today).sum()
future_date_count

3024

In [9]:
old_date = pd.Timestamp("2018-01-01")
old_orders = (df["order_date_parsed"]<old_date).sum()
old_orders

0

### findings

-invalid dates : 0 rows
-future orders : 3079 orders
-orders older than 2018 : 0 orders

### price and revenue consistency
We are trying to confirm if base_price, discounts and final_prices look accurate and also units_sold and revenue_USD.

In [10]:
#Validate whether final_price_usd is correctly derived from base price and discount.

df["final_price_validation"] = (df["base_price_usd"]*(1-(df["discount_percent"]/100)))*df['units_sold']
wrong_price = ((df['revenue_usd']-df['final_price_validation'])>0.1).sum()
wrong_price


0

### findings
number of inconsistent rows : 0

In [11]:
# Revenue Logic & Units Sold Sanity
df['units_sold'].min()
df['units_sold'].max()


4

### findings

Min units sold : 1
Max units sold : 4

In [12]:
# categorical data consistency

df["category"].unique()


array(['Running', 'Lifestyle', 'Basketball', 'Training', 'Gym'],
      dtype=object)

In [13]:
# check for whitepsaces

def has_whitespaces(columns):
    return columns.astype(str).str.strip() != columns.astype(str)

cols = ['brand', 'category', 'gender', 'payment_method', 'sales_channel', 'country','customer_income_level']
mask = df[cols].apply(has_whitespaces)
mask

,brand,category,gender,payment_method,sales_channel,country,customer_income_level
0,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...
29995,False,False,False,False,False,False,False
29996,False,False,False,False,False,False,False
29997,False,False,False,False,False,False,False
29998,False,False,False,False,False,False,False


In [14]:
def sum_of_whitespaces(columns):
    return (columns == True).sum()

whitespace_count = mask[cols].apply(sum_of_whitespaces)
whitespace_count


brand                    0
category                 0
gender                   0
payment_method           0
sales_channel            0
country                  0
customer_income_level    0
dtype: int64

### findings

no white spaces in any columns

In [15]:
#IQR based outlier detection

numeric_cols = ["base_price_usd",
    "discount_percent",
    "final_price_usd",
    "units_sold",
    "revenue_usd",
    "customer_rating"]

def iqr_summary(df,col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr

    outlier_mask = (df[col] < lower) | (df[col] > upper)

    return {
        "column": col,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": outlier_mask.sum(),
        "outlier_pct": outlier_mask.mean() * 100
    }
            
iqr_results = pd.DataFrame([iqr_summary(df,cols) for cols in numeric_cols])
iqr_results

,column,lower_bound,upper_bound,outlier_count,outlier_pct
0,base_price_usd,-20.000,300.000,0,0.00
1,discount_percent,-17.500,42.500,0,0.00
2,final_price_usd,-17.900,256.500,0,0.00
3,units_sold,-3.500,8.500,0,0.00
4,revenue_usd,-229.125,799.875,231,0.77
5,customer_rating,2.000,6.000,0,0.00


### findings

There are 231 outliers only in revenue_usd and it looks okay when checked

In [16]:
# check for unreasonable discounts 

def check_discount(df,columns): 
    error_cols_0 = (df[columns]<0).sum()
    error_cols_70 = (df[columns]>70).sum()
    return {
        "column" : columns,
        "less_than_0" : error_cols_0,
        "greater_than_70" : error_cols_70
    }

cols_disc = ["discount_percent"]
discount_errors = pd.DataFrame(check_discount(df,cols_disc))
discount_errors

,column,less_than_0,greater_than_70
discount_percent,discount_percent,0,0


### Section-2

##Q1

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   order_id                30000 non-null  object        
 1   order_date              30000 non-null  object        
 2   brand                   30000 non-null  object        
 3   model_name              30000 non-null  object        
 4   category                30000 non-null  object        
 5   gender                  30000 non-null  object        
 6   size                    30000 non-null  int64         
 7   color                   30000 non-null  object        
 8   base_price_usd          30000 non-null  int64         
 9   discount_percent        30000 non-null  int64         
 10  final_price_usd         30000 non-null  float64       
 11  units_sold              30000 non-null  int64         
 12  revenue_usd             30000 non-null  float6

In [18]:
#Computing the following KPIs for the entire dataset:

#Total revenue
total_revenue = df["revenue_usd"].sum()
total_revenue

#Total units sold
total_units_sold = df["units_sold"].sum()
total_units_sold

#Average order value (AOV)
number_of_orders = df["order_id"].nunique()
aov = total_revenue / number_of_orders

#Average discount
average_discount = df['discount_percent'].mean()
average_discount

core_business_kpi = pd.DataFrame(
    {
        "metric" : ["total_revenue","total_units_sold","aov","average_discount"],
        "value" : [total_revenue,total_units_sold,aov,average_discount]
    }
)
    
core_business_kpi["value"] = core_business_kpi["value"].astype(int)
core_business_kpi

,metric,value
0,total_revenue,9081448
1,total_units_sold,75006
2,aov,302
3,average_discount,13


##Q2

In [19]:
#Aggregate the data by brand and compute:


brand_kpi = (
    df.groupby("brand")
    .agg(
        total_revenue=("revenue_usd","sum"),
        total_units_sold=("units_sold", "sum"),
        number_of_orders=("order_id", "nunique"),
        average_discount=("discount_percent", "mean")
    )
)

brand_kpi["aov"] = brand_kpi["total_revenue"]/brand_kpi["number_of_orders"]

brand_kpi = brand_kpi.sort_values(by = "total_revenue", ascending = False)
brand_kpi


,total_revenue,total_units_sold,number_of_orders,average_discount,aov
brand,,,,,
ASICS,1561462.50,12874,5132,13.302806,304.260035
Nike,1524582.10,12481,5018,13.190514,303.822658
New Balance,1511401.50,12541,4982,13.269771,303.372441
Puma,1499094.90,12315,4924,13.539805,304.446568
Reebok,1498640.90,12469,5026,13.218265,298.177656
Adidas,1486266.55,12326,4918,13.479057,302.209547


##Q3

In [20]:
#Category-Level Analysis

category_level_analysis = (
    df.groupby("category")
    .agg(
        total_revenue = ("revenue_usd","sum"),
        total_units = ("units_sold","sum"),
        total_orders = ("order_id","nunique")
    )
)
category_level_analysis["aov"] = category_level_analysis["total_revenue"]/category_level_analysis["total_orders"]
category_level_analysis["asp"] = category_level_analysis["total_revenue"]/category_level_analysis["total_units"]
category_level_analysis = category_level_analysis.sort_values(by = "total_revenue", ascending = False)
category_level_analysis

,total_revenue,total_units,total_orders,aov,asp
category,,,,,
Lifestyle,1844628.75,15129,6059,304.444422,121.926681
Training,1836338.65,15142,6018,305.141019,121.274511
Basketball,1822369.65,15160,6074,300.027931,120.209080
Running,1805450.70,14928,5983,301.763446,120.943911
Gym,1772660.70,14647,5866,302.192414,121.025514


In [21]:
df.head()

,order_id,order_date,brand,model_name,category,gender,size,color,base_price_usd,discount_percent,final_price_usd,units_sold,revenue_usd,payment_method,sales_channel,country,customer_income_level,customer_rating,order_date_parsed,final_price_validation
0,ORD100000,2021-01-30,ASICS,Model-370,Running,Unisex,8,Black,162,15,137.70,1,137.7,Card,Retail Store,Germany,Low,4.6,2021-01-30,137.7
1,ORD100001,2026-10-05,Reebok,Model-314,Lifestyle,Men,8,Grey,80,5,76.00,3,228.0,Card,Online,USA,Low,3.9,2026-10-05,228.0
2,ORD100002,2023-11-12,ASICS,Model-763,Lifestyle,Men,8,Black,176,15,149.60,4,598.4,Cash,Retail Store,India,Medium,3.0,2023-11-12,598.4
3,ORD100003,2026-08-29,Reebok,Model-905,Basketball,Women,7,White,61,15,51.85,2,103.7,Card,Retail Store,India,High,3.4,2026-08-29,103.7
4,ORD100004,2019-11-09,Nike,Model-413,Training,Men,11,Black,80,0,80.00,4,320.0,Cash,Online,USA,Medium,3.0,2019-11-09,320.0


In [36]:
country_level_analysis = (
    df.groupby("country")
    .agg(
        total_revenue = ("revenue_usd","sum"),
        total_units = ("units_sold","sum"),
        total_orders = ("order_id", "nunique"),
        avg_discount = ("discount_percent","mean")
    )
)
country_level_analysis["aov"] = country_level_analysis["total_revenue"]/country_level_analysis["total_orders"]
country_level_analysis["asp"] = country_level_analysis["total_revenue"]/country_level_analysis["total_units"]
country_level_analysis = country_level_analysis.sort_values(by = "total_revenue", ascending = False)
country_level_analysis

,total_revenue,total_units,total_orders,avg_discount,aov,asp
country,,,,,,
UAE,1546442.55,12818,5118,13.390973,302.157591,120.646166
UK,1532300.05,12662,5058,13.571570,302.945838,121.015641
India,1520898.95,12539,4991,13.396113,304.728301,121.293480
USA,1511747.45,12472,5027,13.347921,300.725572,121.211309
Germany,1503894.85,12452,4965,13.117825,302.899265,120.775365
Pakistan,1466164.60,12063,4841,13.157405,302.863995,121.542286
